In [ ]:
import pandas as pd
from dask.array import average
from gototile.grid import SkyGrid
from goto_sim.scheduling import HEATSurvey
from goto_sim.utils import load_tilelist
from astropy.time import Time, TimeDelta
import numpy as np
import matplotlib.pyplot as plt
import glob
import matplotlib.font_manager

from notebook.compare_surveys import mean_cadence

plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
flist = matplotlib.font_manager.findSystemFonts()
names = [matplotlib.font_manager.FontProperties(fname=fname).get_name() for fname in flist]
print(names)

In [ ]:
data_dir = "sims"
figure_dir = "figures"
file_pattern = f"{data_dir}/simulation_results_reserve_*.csv"
file_list = glob.glob(file_pattern)
if not file_list:
    raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
print(f"Found {len(file_list)} files matching the pattern.")
reserve_fractions = [float(x .split('_')[-1].split('.csv')[0]) for x in file_list]
results = {}
for file_path, reserve_fraction in zip(file_list, reserve_fractions):
    df = pd.read_csv(file_path, comment='#')
    results[reserve_fraction] = df
results_df = pd.concat(results, names=['reserve_fraction', 'index']).reset_index(level='reserve_fraction')

heats_survey = HEATSurvey()
heats_tiles = heats_survey.tiles
cold_survey = heats_survey.generate_colds()
cold_tiles = cold_survey.get_tiles()
grid: SkyGrid = SkyGrid.from_name("GOTO")


In [ ]:
# Replace HEATS target tiles with HEATS, COLDs tiles with COLDs.
# This is fairly slow, probably a faster way to do this with mapping
results_df['target'] = results_df['target'].replace({tile: 'HEATS' for tile in heats_tiles})
results_df['target'] = results_df['target'].replace({tile: 'COLDs' for tile in cold_tiles})
results_df

In [ ]:
results_df['target'].value_counts()

In [ ]:
cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.median(np.diff(np.sort(x)))
).sort_index()
print(f"Total results: {len(cadence)}")
cold_cadences = cadence.loc[cold_tiles]
print(f"COLD results: {len(cold_cadences)}")

heats_cadences = cadence.loc[heats_tiles]
print(f"HEATS results: {len(heats_cadences)}")


In [ ]:
mean_cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.mean(np.diff(np.sort(x)))).sort_index()
mean_cadence.loc[heats_tiles,0.00].hist()
plt.show()
cadence.loc[heats_tiles,0.00].hist()
plt.show()
modal_cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: pd.Series(np.diff(np.sort(x))).mode()[0]
).sort_index()
modal_cadence.loc[heats_tiles,0.00].hist()
plt.show()



In [ ]:
per_tile_time_delays = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.diff(np.sort(x)))
per_tile_time_delays.loc[heats_tiles,0.00].hist(range=(0,3))

In [ ]:
average_cold_cadence = cold_cadences.groupby('reserve_fraction').mean()
average_heats_cadence = heats_cadences.groupby('reserve_fraction').mean()
plt.figure(figsize=(10, 6))
plt.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
plt.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
plt.xlabel('Fraction of Array Time Reserved for Other Surveys')
plt.ylabel('Median Cadence (Days)')
plt.title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
plt.yticks(minor=True, ticks=np.arange(0, 25, 1))
plt.ylim(0, 25)
plt.legend()
plt.grid()
plt.savefig(f"{figure_dir}/average_cadence_vs_reserve_fraction.png", dpi=300)
plt.show()

In [ ]:
# Print a table of the results for COLD survey
print("COLD Survey Average Cadence:")
for i, row in average_cold_cadence.items():
    print(f"Reserve Fraction (total array): {i/2:.2f}, Average Cadence: {row:05.2f} days, Reserve Fraction (of COLDS): {i:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
# Plot fraction of array time for COLDS survey
ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
ax.legend()

#ax.set_yscale("log")
#ax.yaxis.set_major_locator(plt.LogLocator(base=5))
ax.yaxis.set_major_formatter(plt.ScalarFormatter())
ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
plt.ylim(0, 25)
plt.legend()
plt.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
# Plot fraction of array time for COLDS survey
ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
ax.legend()

#ax.set_yscale("log")
#ax.yaxis.set_major_locator(plt.LogLocator(base=5))
ax.yaxis.set_major_formatter(plt.ScalarFormatter())
ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
plt.ylim(2.5, 10)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Bar chart of number of observations per target for each reserve fraction
observations_per_target = results_df.groupby(['reserve_fraction', 'target']).size().unstack(fill_value=0)
observations_per_target = observations_per_target.iloc[:, [1, 3, 2, 0]]
observations_per_target

In [ ]:
fraction_per_target = observations_per_target.div(observations_per_target.sum(axis=1), axis=0)
fraction_per_target
array_fraction_per_target = fraction_per_target.copy()
array_fraction_per_target.index = array_fraction_per_target.index / 2

In [ ]:
ax = array_fraction_per_target.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=['tab:red', 'tab:grey', 'tab:green', 'tab:blue'],
    alpha=1
)
ax.set_xlabel('Fraction of Array Time Reserved for Other Surveys')
ax.set_ylabel('Fraction of Observations')
ax.set_title('Fraction of Observations per Target vs Reserve Fraction')
for container in ax.containers:
    ax.bar_label(container, fmt='{:.0%}', label_type='center', fontsize=5, color='white')
plt.legend(title='Target', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{figure_dir}/array_fraction_barchart.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
# Plot fraction of array time for COLDS survey
ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
ax.legend()
plt.yticks(minor=True, ticks=np.arange(0, 25, 1))
plt.ylim(0, 25)
plt.legend()
plt.grid()

bar_axis = ax.twinx()
bar_width = 0.02
array_fraction_per_target.plot(
    kind='bar',
    stacked=True,
    ax=bar_axis,
    width=bar_width,
    color=['tab:red', 'tab:grey', 'tab:green', 'tab:blue'],
    legend=False,
    alpha=0.3
)
bar_axis.set_ylabel('Fraction of Observations')
bar_axis.set_ylim(0, 1)

plt.show()

In [ ]:
observations_per_tile = results_df.groupby(['reserve_fraction', 'tile']).size()
observations_per_tile = observations_per_tile.reindex(grid.tilenames, level='tile', fill_value=0)
observations_per_tile

In [ ]:
cadence.loc[heats_tiles,0.00]

In [ ]:
for (reserve_fraction, group), (reserve_fraction, observations) in zip(cadence.groupby('reserve_fraction'), observations_per_tile.groupby('reserve_fraction')):
    if reserve_fraction == 0:
        print(len(observations.values))
        print(group.values)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x=array_fraction_per_target['reserve_fraction'], y=array_fraction_per_target['colds_cadence'], s=1)

In [ ]:
# for each reserve fraction, plot the cadence and total number of observations for each tile
from matplotlib.gridspec import GridSpec
for (reserve_fraction, group_cadence) in cadence.groupby('reserve_fraction'):
    group_observations = observations_per_tile.loc[reserve_fraction]
    heats_observations = group_observations.loc[heats_tiles]
    cold_observations = group_observations.loc[cold_tiles]
    print(f"Reserve Fraction: {reserve_fraction/2:.2f}")
    if True:
        fig = plt.figure(figsize=(12, 8), dpi=100)
        fig.suptitle(f'Reserve Fraction: {reserve_fraction/2:.2f}')
        gs = GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

        # -------------------------
        axes = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
        axes.grid()

        tiles = grid.plot_tiles(axes, array=group_observations.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)
        count_ticks = np.arange(round(min(group_observations.values)-10, -1) if min(group_observations)> 20 else 0, max(group_observations.values) + 1, round(max(group_observations.values)//5, -1) if max(group_observations.values) > 100 else 2)
        fig.colorbar(
            tiles, ax=axes, ticks=count_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label=f'Total number of observations'
            f'\n(min={min(group_observations.values):.0f}, '
            f'mean={np.mean(group_observations.values):.1f}, '
            f'max={max(group_observations.values):.0f})'
        )
        # ------------------------
        axes = fig.add_subplot(gs[1, 0], projection='astro hours mollweide')
        axes.grid()
        tiles = grid.plot_tiles(axes, array=group_cadence.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=np.array(grid.tilenames)[np.array(group_cadence.values) == 0],
                        fc='0.4', ec='none', zorder=0.7)

        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)

        cadence_ticks = np.arange(0, int(max(group_cadence.values))+2, (max(group_cadence.values)//5 if max(group_cadence.values) > 5 else 1))
        fig.colorbar(
            tiles, ax=axes, ticks=cadence_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label='Median days between observations'
            f'\n(min={min(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'mean={np.mean(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'max={max(group_cadence.values):.1f})'
        )
        # -------------------------
        axes = fig.add_subplot(gs[0, 1])
        counts, bins, patches = axes.hist([heats_observations, cold_observations],
                bins=np.arange(0, int(max(group_observations.values))+1, 10), stacked=True,
                color=['red', 'blue'],
                label=['Tiles in the HEATS survey (n={})'.format(len(heats_tiles)),
                       'Tiles in the COLD survey (n={})'.format(len(cold_tiles))],
        )
        axes.legend(loc='upper right', bbox_to_anchor=(1, 1.25), frameon=False)
        axes.set_xlabel('Number of observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(int(min(group_observations.values))-1, int(max(group_observations.values))+1+0.5)
        axes.set_xticks(count_ticks)
        # -------------------------
        axes = fig.add_subplot(gs[1, 1])
        #axes.hist([median_cadence[heats_mask]*24, median_cadence[~heats_mask]*24],
        axes.hist([group_cadence.loc[heats_tiles], group_cadence.loc[cold_tiles]],
                bins=np.arange(0, max(group_cadence.values)+1), stacked=True,
                color=['red', 'blue'],
                #label=['Tiles in the HEATS survey', 'Tiles in the COLD survey'],
        )
        #axes.legend()
        axes.set_xlabel('Days between observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(0-0.1, int(max(group_cadence.values))+2)
        axes.set_xticks(cadence_ticks)
        plt.savefig(f"{figure_dir}/reserve_fraction_{reserve_fraction/2:.2f}_summary.png", dpi=300)
        plt.show()
